<a href="https://colab.research.google.com/github/tekpinar/gromacscolab/blob/main/MD_TrajectoryAnalysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# GROMACSCOLAB: Trajectory Analysis Notebook

**Molecular Dynamics** (**MD**) simulation is a powerful computational method used to understand the atomic-level behavior of biomolecules and to study the thermodynamic and kinetic properties of systems. **GROMACS**, with its high-performance computing capabilities and flexible user interface, is widely preferred for performing MD simulations. In this notebook, we will perform standard analyses of a molecular dynamics simulation. **You will need to upload a trajectory containing only the protein to conduct this analysis.**

🔷 Workflow Overview

This notebook guides you step by step for analyzing a protein for molecular dynamics (MD) simulation using GROMACS.

You will:

* Upload a clean -protein only- structure (typically .pdb or .gro file)
* Upload a clean -protein only trajectory (typically .xtc or .trr file)

* You will perform the following analyses in this notebook:
  * RMSD
  * RMSF
  * Rg
  * PCA
  * SASA
  * Secondary Structure
  * Structural Clustering




In [ ]:
#@title Install GROMACS
# `apt install` without -y aborts in a non-interactive notebook shell.
!apt-get update -qq
!apt-get install -y -qq gromacs
import subprocess
print(subprocess.run(['gmx','--version'],capture_output=True,text=True).stdout.splitlines()[0])


In [ ]:
#@title Install the Python libraries
# mkdssp is NOT needed: MDAnalysis.analysis.dssp is a pure-Python
# reimplementation and never shells out to the DSSP binary.
!pip install -q py3Dmol "mdanalysis>=2.8" biopython kaleido==0.2.1 scikit-learn
import MDAnalysis; print("MDAnalysis", MDAnalysis.__version__)


In [ ]:
#@title Upload your PDB and trajectory file containing only protein! {run: "auto"}
from google.colab import files

print("Upload your PDB file:")
uploaded_pdb = files.upload()
pdb_filename = list(uploaded_pdb.keys())[0]

print("\nUpload your trajectory file (xtc, trr):")
uploaded_traj = files.upload()
traj_filename = list(uploaded_traj.keys())[0]

print(f"\nPDB file uploaded: {pdb_filename}")
print(f"Trajectory file uploaded: {traj_filename}")

In [ ]:
#@title Check the inputs and set up named group selection
# The previous version hard-coded index numbers (3 = C-alpha, 2 = Protein-H,
# 1 = Protein), which are only correct for a protein-only system with the
# default index. GROMACS accepts group NAMES at the same prompts, which is
# both robust and self-documenting, so we use those throughout.
import subprocess, os, MDAnalysis as mda

GRP_FIT  = "C-alpha"   # fitting / RMSD / PCA
GRP_PROT = "Protein"   # whole protein
GRP_OUT  = "Protein"   # output group

u = mda.Universe(pdb_filename, traj_filename)
prot = u.select_atoms("protein")
chains = sorted(set(getattr(prot.atoms, "chainIDs", ["A"]*prot.n_atoms)))

print(f"Reference : {pdb_filename}  ({u.atoms.n_atoms} atoms)")
print(f"Trajectory: {traj_filename}  ({len(u.trajectory)} frames, "
      f"{u.trajectory.totaltime/1000:.2f} ns)")
print(f"Protein   : {prot.n_atoms} atoms, {prot.n_residues} residues, "
      f"chain(s) {chains}")

non_protein = u.atoms.n_atoms - prot.n_atoms
if non_protein:
    print(f"\nWARNING: {non_protein} non-protein atoms present. This notebook "
          "expects the protein-only trajectory produced by the PBC-correction "
          "cell at the end of MD_Execution (md_protein.pdb / md_protein.xtc).")


This command enables **interactive widget support** in Google Colab, allowing **real-time visualization and analysis** of GROMACS simulation data.

In [ ]:
from google.colab import output
output.enable_custom_widget_manager()

# Root Mean Square Deviation (RMSD)
### Description:
RMSD measures the structural deviations of a molecule over time. RMSD reflects the overall change of the entire structure relative to its initial conformation. RMSD equation is given below:

$$
\text{RMSD}(t) = \sqrt{ \frac{1}{N} \sum_{i=1}^{N} \left( r_i(t) - r_i^{\text{ref}} \right)^2 }
$$

### Purpose:
RMSD is used to evaluate the stability of the protein or molecule, check whether the system has reached equilibrium, and track significant structural changes over time.


**Here:**

*   $r_i(t)-$  position of atom $i$ at time $t$
*   $r_i^{\text{ref}}$ → reference (initial) position of atom $i$
*   $N -$  total number of atoms



In [ ]:
#@title Calculate RMSD
!printf "%s\n%s\n" "${GRP_FIT}" "${GRP_FIT}" | gmx rms -s "{pdb_filename}" -f "{traj_filename}" -o rmsd.xvg -tu ns
import os; assert os.path.exists("rmsd.xvg"), "gmx rms failed - see output above"

In [ ]:
# # If you want a more Pythonic way to call external programs,
# # you can uncomment and use this version!
# import subprocess
# try:
#     pdb_filename
#     traj_filename
# except NameError:
#     raise NameError("pdb_filename or traj_filename is undefined!")

# rmsdCmd = [
#     "gmx", "rms",
#     "-s", pdb_filename,
#     "-f", traj_filename,
#     "-o", "rmsd.xvg",
#     "-tu", "ns"
# ]

# input_text = "3\n3\n"

# process = subprocess.Popen(rmsdCmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
# stdout, stderr = process.communicate(input_text.encode())

# print(stdout.decode())
# print(stderr.decode())

# if process.returncode == 0:
#     print("RMSD analysis finished successfully.")
# else:
#     print("RMSD analysis failed!")

### Plot RMSD

This plot reads RMSD data from an XVG file and provides an interactive visualization of structural deviations of the protein over time. It helps to easily track the overall stability and conformational changes of the protein during the simulation.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def read_xvg(filename):
  """Reads an XVG file using NumPy."""
  data = np.loadtxt(filename, comments=['@', '#'], dtype=float)
  return data

# Assuming your rmsd data is in a file named 'rmsd.xvg'
rmsd_data = read_xvg('rmsd.xvg')

# Extract x and y values from the data.  Adjust column indices if necessary.
time = rmsd_data[:, 0]
rmsd = rmsd_data[:, 1]


# Create an interactive plot using Plotly
fig = go.Figure(data=go.Scatter(x=time, y=rmsd, mode='lines+markers'))
fig.update_layout(
    title='RMSD Plot',
    xaxis_title='Time (ns)',  # Update x-axis label
    yaxis_title='RMSD (nm)',
    # xaxis=dict(range=[0.0, 0.5])
)

fig.show()
# Save figure to a file
fig.write_image("rmsd.png")


# Root Mean Square Fluctuation (RMSF)

### Description:
RMSF is a measure of how much each atom or residue deviates from its average position during a molecular dynamics simulation, providing information about flexibility. RMSF equation is given below:

$$
\text{RMSF}_i = \sqrt{\frac{1}{T} \sum_{t=1}^{T} \left( r_i(t) - \langle r_i \rangle \right)^2 }
$$

**where:**


*   $r_i(t)-$  position of atom $i$ at time $t$
*   $\langle r_i \rangle -$ average position of atom $i$ over the simulation
*   $T -$  total number of time frames


### Purpose:
The Root Mean Square Fluctuation (RMSF) is calculated to measure the mobility and flexibility of each atom within the protein structure over time. This metric highlights the fluctuations of atoms in different structural regions, providing insights into the dynamic behavior of the protein.

In [ ]:
#@title Calculate RMSF (per residue)
!echo "{GRP_PROT}" | gmx rmsf -s "{pdb_filename}" -f "{traj_filename}" -res -fit -o rmsf.xvg -oq rmsf.pdb
import os; assert os.path.exists("rmsf.xvg"), "gmx rmsf failed - see output above"


In [ ]:
import numpy as np, pandas as pd, MDAnalysis as mda

# The previous version inferred chain boundaries from the RMSF file alone, by
# assuming residue numbering resets to exactly 1 at the start of each chain.
# That fails for continuously numbered or non-1-based structures. We take the
# chain assignment from the structure file instead, which is authoritative.
rmsf_raw = np.loadtxt('rmsf.xvg', comments=['@','#'])
resid_xvg, rmsf_vals = rmsf_raw[:,0].astype(int), rmsf_raw[:,1]

u_ref = mda.Universe(pdb_filename)
res = u_ref.select_atoms("protein").residues
try:
    chain_of = [a for a in res.atoms.chainIDs[
        np.unique(res.atoms.resindices, return_index=True)[1]]]
except Exception:
    chain_of = None

if chain_of is not None and len(chain_of) == len(resid_xvg):
    chains = chain_of
elif len(res) == len(resid_xvg):
    chains = [getattr(r.atoms[0], "chainID", "A") or "A" for r in res]
else:
    print(f"WARNING: {len(resid_xvg)} RMSF rows vs {len(res)} residues in "
          f"{pdb_filename}; treating the protein as a single chain.")
    chains = ["A"] * len(resid_xvg)

rmsf_df_processed = pd.DataFrame({"ChainID": chains,
                                  "ResidueID": resid_xvg,
                                  "RMSF": rmsf_vals})

chain_info = (rmsf_df_processed.groupby("ChainID")
              .agg(StartResidue=("ResidueID","min"),
                   EndResidue=("ResidueID","max"),
                   NumResidues=("ResidueID","count"))
              .reset_index())
print(f"{len(rmsf_df_processed)} residues with RMSF data")
print(chain_info.to_string(index=False))


In [ ]:
import plotly.graph_objects as go
import pandas as pd # Ensure pandas is imported

# Define a list of colors and symbols for different chains
colors = ['blue', 'red', 'green', 'purple', 'orange', 'cyan', 'magenta', 'brown', 'pink', 'gray']
symbols = ['circle', 'square', 'diamond', 'cross', 'x', 'triangle-up', 'pentagon', 'hexagram', 'star', 'hourglass']

# Create an interactive plot using Plotly
fig = go.Figure()

# Iterate through each unique chain in the processed DataFrame
for i, (chain_id, chain_data) in enumerate(rmsf_df_processed.groupby('ChainID')):
    fig.add_trace(go.Scatter(
        x=chain_data['ResidueID'],
        y=chain_data['RMSF'],
        mode='lines+markers',
        name=f'Chain {chain_id}',
        marker=dict(symbol=symbols[i % len(symbols)], color=colors[i % len(colors)]),
        line=dict(color=colors[i % len(colors)])
    ))

fig.update_layout(
    title='RMSF Plot per Chain',
    xaxis_title='Residue Number',
    yaxis_title='RMSF (nm)',
    hovermode='x unified' # Provides a unified hover experience across traces
)

fig.show()
fig.write_image("rmsf.png")


### Display 3D Structure Colored by RMSF (B-factor)

This code loads a PDB file (rmsf.pdb) and visualizes the protein in 3D using py3Dmol. It colors the protein according to B-factor values (corresponding to RMSF), using a rainbow gradient to highlight flexibility. The minimum and maximum B-factor values are automatically extracted from the file. The view is centered and zoomed, with labels and a legend showing the B-factor range. This allows identifying flexible and rigid regions and interpreting atomic mobility interactively.

In [ ]:
#@title Display 3D structure of protein colored according to RMSF values {run: "auto"}
import py3Dmol
import pandas as pd
import re

def get_bfactor_range(pdb_file):
    """
    Extract B-factor values from PDB file and return their range

    Parameters:
    pdb_file (str): Path to the PDB file

    Returns:
    tuple: (minimum B-factor, maximum B-factor)
    """
    bfactors = []

    with open(pdb_file, 'r') as f:
        for line in f:
            if line.startswith('ATOM') or line.startswith('HETATM'):
                try:
                    # B-factor is typically in columns 61-66
                    bfactor = float(line[60:66].strip())
                    bfactors.append(bfactor)
                except (ValueError, IndexError):
                    continue

    if not bfactors:
        return (0, 100)  # default range if no B-factors found

    return (min(bfactors), max(bfactors))

def visualize_protein_bfactor(pdb_file):
    """
    Visualize protein structure in cartoon representation colored by B-factor
    using automatically determined range and rainbow colors

    Parameters:
    pdb_file (str): Path to the PDB file
    """

    # Create a py3Dmol view instance
    view = py3Dmol.view()

    # Get B-factor range from the file
    bfactor_min, bfactor_max = get_bfactor_range(pdb_file)

    # Load the PDB file
    with open(pdb_file, 'r') as f:
        pdb_data = f.read()

    # Add the molecule to the viewer
    view.addModel(pdb_data, "pdb")

    # Set cartoon representation with rainbow coloring based on B-factor
    view.setStyle({'cartoon': {
        'colorscheme': {
            'prop': 'b',
            'gradient': 'linear',  # Using rainbow color scheme
            'min': bfactor_min,
            'max': bfactor_max,
            'colors': ["blue", "white", "red"]
        }
    }})

    # Center and zoom the view
    view.zoomTo()

    # Add legend for B-factor coloring
    view.addPropertyLabels(
        prop='b',
        gradient='bwr',
        min=bfactor_min,
        max=bfactor_max,
        legend={'x': 0.85, 'y': 0.5}
    )

    # Add text showing the B-factor range
    view.addLabel(f"B-factor range: {bfactor_min:.2f} - {bfactor_max:.2f}",
                 {'position': {'x': -20, 'y': -20, 'z': 0},
                  'backgroundColor': 'white',
                  'fontColor': 'black'})

    return view

# Replace with your PDB file path
pdb_file = "rmsf.pdb"
view = visualize_protein_bfactor(pdb_file)
view.show()

# Optional: Save the visualization as HTML
# view.save('protein_visualization.html')

# Radius of Gyration (Rg) Calculation

### Description:
The radius of gyration is a measure of the distribution of atoms around the protein’s center of mass, indicating how compact or extended the structure is during the simulation. Radius of Gyration (Rg) equation is given below:

$$
R_g = \sqrt{\frac{\sum_{i=1}^{N} m_i \, \| r_i - r_{com} \|^2}{\sum_{i=1}^{N} m_i}}
$$

**where:**  

- $m_i →$  mass of atom i  
- $r_i →$  position of atom i  
- $r_{\text{com}} →$ center of mass of the protein  
- $N →$ total number of atoms


In [ ]:
#@title Calculate the radius of gyration
!echo "{GRP_PROT}" | gmx gyrate -s "{pdb_filename}" -f "{traj_filename}" -o gyrate.xvg
import os; assert os.path.exists("gyrate.xvg"), "gmx gyrate failed - see output above"


### Plot Radius of Gyration

Reads the gyration data file and creates an interactive plot showing the protein's radius of gyration over time. This allows monitoring the compactness of the protein and observing structural changes, folding, or unfolding events during the simulation.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Read the gyrate.xvg file
data = np.loadtxt('gyrate.xvg', comments=['@', '#'])

# Extract time and gyration radius
time = data[:, 0]
gyration_radius = data[:, 1]

# Create the plot
fig = go.Figure(data=go.Scatter(x=time/1000.0, y=gyration_radius, mode='lines+markers'))
fig.update_layout(
    title='Radius of Gyration',
    xaxis_title='Time (ns)',
    yaxis_title='Radius of Gyration (nm)'
)
fig.show()
fig.write_image("radius-of-gyration.png")


# PCA Analysis

  Calculates the covariance matrix of atomic coordinates to perform Principal Component Analysis  (**PCA**) on protein motions. This helps to identify the dominant modes of motion, reduce dimensionality, and highlight collective structural changes in the protein during the simulation.

In [ ]:
#@title Covariance matrix for PCA
!printf "%s\n%s\n" "${GRP_FIT}" "${GRP_FIT}" | gmx covar -s "{pdb_filename}" -f "{traj_filename}" \
    -o eigenval.xvg -v eigenvec.trr -av average.pdb
import os; assert os.path.exists("eigenval.xvg"), "gmx covar failed - see output above"


### PCA Eigenvalues Plot (Interactive)

This code loads eigenvalues from the `eigenval.xvg `file and creates an interactive scatter plot using Plotly. The plot visualizes the variance captured by each principal component, helping to identify which components dominate the protein’s collective motions.

In [ ]:
import numpy as np, plotly.graph_objects as go

eig = np.loadtxt('eigenval.xvg', comments=['#','@'])
idx, val = eig[:,0], eig[:,1]
pct = 100 * val / val.sum()
cum = np.cumsum(pct)
n_show = min(20, len(idx))

fig = go.Figure()
fig.add_trace(go.Bar(x=idx[:n_show], y=pct[:n_show], name='variance explained (%)'))
fig.add_trace(go.Scatter(x=idx[:n_show], y=cum[:n_show], name='cumulative (%)',
                         mode='lines+markers', yaxis='y2'))
fig.update_layout(title='Contributions of principal components of the C-alpha covariance matrix',
                  xaxis_title='Eigenvector index',
                  yaxis_title='Variance explained (%)',
                  yaxis2=dict(title='Cumulative (%)', overlaying='y',
                              side='right', range=[0,100]),
                  hovermode='x unified')
fig.show()
fig.write_image("pca-eigenvalues.png", scale=2)

pc_variance = pct
print("  PC   eigenvalue (nm^2)   variance %   cumulative %")
for i in range(min(5, len(idx))):
    print(f"  {int(idx[i]):<4} {val[i]:>16.4f} {pct[i]:>12.2f} {cum[i]:>14.2f}")


### Perform PCA Projections

Runs GROMACS `anaeig` to project the trajectory onto the selected range of principal components, generating a 2D projection plot. This allows visualization of collective motions in the protein along the chosen principal components.

In [ ]:
#@title 2-D projections and extreme structures along each PC
# `-extr` names its output extreme1.pdb, extreme2.pdb, ... by position within
# the -first..-last range, not by PC index, and the previous nested loop
# overwrote them on every iteration - so the viewer below showed whichever
# mode happened to be written last. We generate the extremes one PC at a time
# into uniquely named files, and the 2-D projections separately.
import os, subprocess

for pc in (1, 2, 3):
    !printf "%s\n%s\n" "${GRP_FIT}" "${GRP_FIT}" | gmx anaeig -s "{pdb_filename}" -f "{traj_filename}" \
        -first {pc} -last {pc} -extr extremePC{pc}.pdb -nframes 20 >/dev/null 2>&1

for i in (1, 2, 3):
    for j in range(i + 1, 4):
        !printf "%s\n%s\n" "${GRP_FIT}" "${GRP_FIT}" | gmx anaeig -s "{pdb_filename}" -f "{traj_filename}" \
            -first {i} -last {j} -2d 2dproj_{i}vs{j}.xvg >/dev/null 2>&1

for f in sorted(os.listdir('.')):
    if f.startswith(('extremePC','2dproj_')):
        print(f"{f:<24} {os.path.getsize(f):>10,} bytes")


In [ ]:
playPrincipalComponent = "2" #@param ["1", "2", "3"]


In [ ]:
import py3Dmol, os

fn = f"extremePC{playPrincipalComponent}.pdb"
assert os.path.exists(fn), f"{fn} not found - run the anaeig cell above."

v = py3Dmol.view(width=800, height=600)
v.addModelsAsFrames(open(fn).read())
v.setStyle({'cartoon': {'color': 'spectrum'}})
v.animate({'loop': 'backAndForth', 'interval': 80})
v.zoomTo()
print(f"Motion along PC{playPrincipalComponent} "
      f"({pc_variance[int(playPrincipalComponent)-1]:.1f}% of the total variance)")
v.show()


In [ ]:
#@title Select Principal Components to plot
firstComponent = "1" #@param ["1", "2", "3"]
lastComponent = "2" #@param ["1", "2", "3"]

!echo $firstComponent
!echo $lastComponent

### 2D PCA Projection Plot

Reads the **2D PCA** projection file and creates an interactive scatter plot with color mapping to visualize the trajectory along the selected principal components. This allows tracking how the protein moves collectively in the plane defined by the chosen components.

In [ ]:
import plotly.graph_objects as go
import numpy as np

# Read the data from the xvg file
data = np.loadtxt(f"2dproj_{firstComponent}vs{lastComponent}.xvg", comments=['@', '#'])

# Extract x, y values
x = data[:, 0]
y = data[:, 1]

# Create a color scale based on the order of the data points
colors = np.linspace(0, 1, len(x))  # Scale from 0 to 1

#e scatter plot with color mapping
fig = go.Figure(data=go.Scatter(
    x=x,
    y=y,
    mode='markers',
    marker=dict(
        color=colors,
        colorscale='RdBu',  # Use a red-blue color scale
        colorbar=dict(title="Data Point Order")
    )
))

fig.update_layout(
    title='2D Projection of PCA',
    xaxis_title=f'PC{firstComponent}',
    yaxis_title=f'PC{lastComponent}'
)

fig.show()
fig.write_image(f"2dproj-PC{firstComponent}-vs-PC{lastComponent}.png")


### K-Means Clustering on PCA Components

Performs K-Means clustering on the 2D PCA-projected data to group similar protein conformations together.

**K-Means:** An algorithm that partitions data into K clusters, minimizing the distance between points in the same cluster while maximizing the distance between clusters.

**Silhouette Score:** A metric used to determine the optimal number of clusters by measuring how well each point fits within its cluster.

Clustering is important because it helps identify patterns and dominant conformational states in the protein’s motion, revealing groups of similar collective movements along the selected principal components.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
import numpy as np, plotly.graph_objects as go

data_2d = np.loadtxt(f"2dproj_{firstComponent}vs{lastComponent}.xvg", comments=['@','#'])
X = data_2d[:, [0, 1]]

best_score, best_k = -1, 2
for k in range(2, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    s = silhouette_score(X, km.labels_)
    if s > best_score:
        best_score, best_k = s, k

km = KMeans(n_clusters=best_k, random_state=42, n_init=10).fit(X)
labels = km.labels_
print(f"Best number of clusters: {best_k} (silhouette {best_score:.3f})")
for c in range(best_k):
    print(f"   cluster {c}: {np.sum(labels==c):>6} frames "
          f"({100*np.mean(labels==c):.1f}%)")

fig = go.Figure(go.Scatter(
    x=X[:,0], y=X[:,1], mode='markers',
    marker=dict(color=labels, colorscale='viridis',
                colorbar=dict(title="Cluster"), size=5)))
# the title was previously a plain string, so it rendered the literal braces
fig.update_layout(
    title=f'K-means clustering in the PC{firstComponent}-PC{lastComponent} plane',
    xaxis_title=f'PC{firstComponent} (nm)',
    yaxis_title=f'PC{lastComponent} (nm)')
fig.show()
fig.write_image(f"kmeans-PC{firstComponent}-vs-PC{lastComponent}.png", scale=2)


# Solvent Accessible Surface Area (SASA) Calculation

Runs GROMACS SASA to calculate the solvent-accessible surface area of the selected group over the trajectory. This provides insights into the exposure of residues to the solvent, helping to understand protein folding, stability, and interactions with other molecules.

In [ ]:
#@title Solvent-accessible surface area
!echo "{GRP_PROT}" | gmx sasa -s "{pdb_filename}" -f "{traj_filename}" -o sasa.xvg -tu ns
import os; assert os.path.exists("sasa.xvg"), "gmx sasa failed - see output above"


### Plot SASA

Reads the SASA output file and creates an interactive plot showing the solvent-accessible surface area of the protein over time. This allows tracking how the protein’s surface exposure changes during the simulation and can indicate folding, unfolding, or conformational changes.

In [ ]:
import numpy as np
import plotly.graph_objects as go

def read_xvg(filename):
  """Reads an XVG file using NumPy."""
  data = np.loadtxt(filename, comments=['@', '#'], dtype=float)
  return data

# Assuming your rmsd data is in a file named 'rmsd.xvg'
rmsd_data = read_xvg('sasa.xvg')

# Extract x and y values from the data.  Adjust column indices if necessary.
time = rmsd_data[:, 0]
rmsd = rmsd_data[:, 1]


# Create an interactive plot using Plotly
fig = go.Figure(data=go.Scatter(x=time, y=rmsd, mode='lines+markers'))
fig.update_layout(
    title='SASA Plot',
    xaxis_title='Time (ns)',  # Update x-axis label
    yaxis_title= '$SASA (nm^2)$',
    # xaxis=dict(range=[0.0, 0.5])
)

fig.show()
fig.write_image("sasa.png")

# Secondary Structure Analysis

Protein secondary structure may not remain identical to the initial conformation during an MD simulation. The purpose of this analysis is to track secondary structure across the MD trajectory.

In [ ]:
#@title Install dssp and test it
#!apt install dssp
#!mkdssp --version


In [ ]:
#@title Calculate secondary structure for every frame
import MDAnalysis as mda
from MDAnalysis.analysis import dssp
import numpy as np

u = mda.Universe(pdb_filename, traj_filename)

# DSSP needs a complete N, CA, C, O backbone for every residue. Build the
# selection in one vectorised pass - the previous version grew an AtomGroup
# with repeated `+=` inside a Python loop, which is O(n^2) in residue count.
prot = u.select_atoms("protein")
need = ("N", "CA", "C", "O")
counts = {nm: {} for nm in need}
for nm in need:
    sel = prot.select_atoms(f"name {nm}")
    for ri in sel.resindices:
        counts[nm][ri] = counts[nm].get(ri, 0) + 1

good = [ri for ri in np.unique(prot.resindices)
        if all(counts[nm].get(ri, 0) == 1 for nm in need)]
dropped = [ri for ri in np.unique(prot.resindices) if ri not in set(good)]

protein_for_dssp_ag = u.atoms[np.isin(u.atoms.resindices, good)]
assert protein_for_dssp_ag.n_atoms, "No residues with a complete backbone."

if dropped:
    print(f"{len(dropped)} residue(s) excluded for incomplete backbone: "
          + ", ".join(str(u.residues[ri].resid) for ri in dropped[:20])
          + (" ..." if len(dropped) > 20 else ""))

dss = dssp.DSSP(protein_for_dssp_ag).run()

# Residue identity of each DSSP column, taken from the AtomGroup actually
# analysed. The previous version hard-coded `[209]` as the excluded residue,
# a leftover from one specific system that silently corrupted the alignment
# for every other protein.
dssp_residues = protein_for_dssp_ag.residues
dssp_resids   = dssp_residues.resids
try:
    dssp_chains = [r.atoms[0].chainID or "A" for r in dssp_residues]
except Exception:
    dssp_chains = ["A"] * len(dssp_resids)

print(f"DSSP: {dss.results.dssp.shape[0]} frames x "
      f"{dss.results.dssp.shape[1]} residues")


In [ ]:
import collections
last = collections.Counter(dss.results.dssp[-1])
print("Secondary structure in the final frame:",
      {k: v for k, v in last.most_common()})


In [ ]:
#@title Define color parameters for secondary structures
import numpy as np

dssp_map = {
    "-": 0,   # Coil
    "H": 1,   # Alpha helix
    "G": 1,   # 3-10 helix
    "I": 1,   # Pi helix
    "E": 2,   # Beta sheet
    "B": 2,   # Beta bridge
    "T": 3,   # Turn
    "S": 3    # Bend
}

labels = {
    0: "Coil",
    1: "Helix",
    2: "Sheet",
    3: "Turn"
}

#ss_num = np.vectorize(dssp_map.get)(ss)
ss_num = np.vectorize(dssp_map.get)(dss.results.dssp)

#Define a discrete colorscale
colorscale = [
    [0.00, "#B3B3B3"], [0.25, "#B3B3B3"],  # Coil
    [0.25, "#D55E00"], [0.50, "#D55E00"],  # Helix
    [0.50, "#0072B2"], [0.75, "#0072B2"],  # Sheet
    [0.75, "#009E73"], [1.00, "#009E73"]   # Turn
]

In [ ]:
import plotly.graph_objects as go
import numpy as np, pandas as pd

# Plot straight from the DSSP result and its own residue list - no dependence
# on the RMSF cell and no hard-coded residue exclusions.
ss_df = pd.DataFrame({"ChainID": dssp_chains, "ResidueID": dssp_resids})
ss_df["col"] = np.arange(len(ss_df))

times_ns = np.array([ts.time for ts in u.trajectory]) / 1000.0
if len(times_ns) != ss_num.shape[0]:
    times_ns = np.arange(ss_num.shape[0])

for chain_id, grp in ss_df.groupby("ChainID"):
    cols = grp["col"].values
    fig = go.Figure(go.Heatmap(
        z=ss_num[:, cols].T,
        x=times_ns,
        y=grp["ResidueID"].values,
        colorscale=colorscale, zmin=0, zmax=3,
        colorbar=dict(tickmode="array", tickvals=[0,1,2,3],
                      ticktext=[labels[i] for i in range(4)],
                      title="Secondary structure")))
    fig.data[0].customdata = dss.results.dssp[:, cols].T
    fig.data[0].hovertemplate = ("Residue %{y}<br>Time %{x:.2f} ns<br>"
                                 "SS: %{customdata}<extra></extra>")
    fig.update_layout(
        title=f"Secondary structure evolution - chain {chain_id}",
        xaxis_title="Time (ns)", yaxis_title=f"Residue (chain {chain_id})",
        yaxis_autorange="reversed", font=dict(size=13), height=650)
    fig.show()
    fig.write_image(f"secondary-structure-chain-{chain_id}.png", scale=2)

    frac = {labels[k]: float(np.mean(ss_num[:, cols] == k)) for k in range(4)}
    print(f"chain {chain_id}: " +
          "  ".join(f"{k} {100*v:.1f}%" for k, v in frac.items()))


# Structural Clustering Analysis using GROMACS (gmx cluster)

### Purpose:

* Groups similar protein structures together, showing which conformations recur during the simulation.

* Identifies dominant conformational states of the protein.

* Helps analyze dynamic behavior and structural stability by examining the number, size, and composition of clusters.

* Simplifies visualization and interpretation by reducing many structures to representative clusters.

### What this code does:

* Uses gmx cluster to group similar protein conformations from a molecular dynamics trajectory.

* The analysis is based on RMSD (Root Mean Square Deviation) between frames.

* A cutoff value (0.5 nm but you can change it below) is used to decide whether two structures are considered part of the same cluster.

**Why it’s useful:**

* Helps identify recurring protein conformations along the trajectory.

* Reveals dominant structural states of the protein.

* Complements PCA/K-Means clustering by using full atomic coordinates instead of PCA-projected 2D data.

### Key points and differences from previous clustering (K-Means on PCA):

1. **Input data:**
    * K-Means: PCA-reduced 2D vectors of protein motions.
    * gmx cluster: Full atomic coordinates from the trajectory.

2. **Clustering method:**

    * K-Means: Partitions data into a fixed number of clusters by minimizing distance to cluster centers.
    * gmx cluster: Groups structures based on RMSD similarity, cutoff defines maximum allowed RMSD within a cluster.

3. **Cutoff (0.25 nm):**

    * Two conformations are in the same cluster if their RMSD ≤ 0.25 nm.
    * Smaller cutoff → more, tighter clusters.
    * Larger cutoff → fewer, broader clusters.

4. **Purpose:**

    * K-Means (PCA) highlights collective motions and main movement directions.
    * gmx cluster identifies structurally similar states and dominant conformations along the trajectory

In [ ]:
#@title If there are large conformational changes, change this parameter to a bigger value!
rmsdCutoff = 0.2 #@param {type:"slider", min:0, max:1.5, step:0.05}
print(f"Selected RMSD cutoff is {rmsdCutoff}")

In [ ]:
import subprocess, os, MDAnalysis as mda
import numpy as np

# gmx cluster with the gromos method builds a full frame-by-frame RMSD matrix
MAX_FRAMES = 500  #@param {type:"integer"}

n_frames = len(mda.Universe(pdb_filename, traj_filename).trajectory)
skip = max(1, int(np.ceil(n_frames / MAX_FRAMES)))
est_mb = (n_frames / skip) ** 2 * 4 / 1e6
print(f"{n_frames} frames; using every {skip}{'st' if skip==1 else 'th'} frame "
      f"-> {n_frames//skip} frames, RMSD matrix approx {est_mb:.0f} MB")

# We remove -dt 0 to let GROMACS read frames sequentially based on -skip
clusteringCmd = ["gmx", "cluster", "-s", pdb_filename, "-f", traj_filename,
                 "-cutoff", str(rmsdCutoff), "-method", "gromos",
                 "-skip", str(skip),
                 "-o", "rmsd-matrix.xpm", "-g", "cluster.log",
                 "-cl", "clusters.pdb"]

p = subprocess.run(clusteringCmd, input=f"{GRP_FIT}\n{GRP_OUT}\n",
                   capture_output=True, text=True)
print(p.stdout[-1500:])
if p.returncode != 0:
    print("STDERR:", p.stderr[-1500:])
    print("Cluster analysis failed!")
else:
    print("Cluster analysis finished successfully.")

### Display Cluster Representative Structures (3D)

This code loads the cluster.pdb file containing representative protein structures from each cluster. Each cluster groups similar conformations observed along the trajectory. The 3D visualization colors each representative differently and labels the clusters, allowing easy identification of typical protein conformations and collective movements. This helps interpret which regions of the protein move together or remain stable

In [ ]:
import os
import py3Dmol

cluster_pdb_filename = "clusters.pdb"

if not os.path.exists(cluster_pdb_filename):
    print(f"'{cluster_pdb_filename}' not found!")
else:
    with open(cluster_pdb_filename, "r") as f:
        pdb_data = f.read()

    models = [m.strip() for m in pdb_data.split("ENDMDL") if m.strip()]

    view = py3Dmol.view(width=800, height=600)
    colors = ["red", "blue", "green", "orange", "purple",
              "cyan", "magenta", "yellow", "brown", "pink"]

    for i, model_data in enumerate(models):
        model_text = model_data
        if not model_text.endswith("ENDMDL"):
            model_text += "\nENDMDL"
        view.addModel(model_text, "pdb")
        view.setStyle({'model': i}, {"cartoon": {"color": colors[i % len(colors)]}})

    view.zoomTo()
    view.show()


In [ ]:
import os

logfile = "cluster.log"

if not os.path.exists(logfile):
    print("cluster.log not found! Make sure that you executed gmx cluster!")
else:
    with open(logfile, "r") as f:
        log_data = f.read()

    print("--- GROMACS Cluster Log Summary ---\n")
    lines = log_data.splitlines()
    found_summary = False
    for line in lines:
        # Match the general summary stats or any individual cluster lines containing '|'
        if "clusters" in line or "cl." in line or "| " in line or "Found" in line:
            print(line)
            found_summary = True

    if not found_summary:
        # Fallback
        print("\n".join(lines[:40]))

In [ ]:
import os, glob

analysis_files = sorted(set(
    glob.glob("*.xvg") + glob.glob("*.png") + glob.glob("*.log") +
    glob.glob("extremePC*.pdb") +
    [f for f in ("rmsf.pdb","clusters.pdb","average.pdb","eigenvec.trr",
                 "rmsd-matrix.xpm") if os.path.exists(f)]))
print(f"{len(analysis_files)} files to archive:")
for f in analysis_files:
    print(f"   {f:<34} {os.path.getsize(f):>12,}")


In [ ]:
import tarfile
import os

archive_name = "analysis_results.tar.gz"

with tarfile.open(archive_name, "w:gz") as tar:
    for f_name in analysis_files:
        if os.path.exists(f_name):
            tar.add(f_name)
            print(f"Added {f_name} to {archive_name}")
        else:
            print(f"Warning: File {f_name} not found and will not be added to the archive.")

print(f"Successfully created {archive_name}")

In [ ]:
#@title Download the results
from google.colab import files
files.download(archive_name)
print("If the download does not start, use the file browser in the left sidebar.")


**Congratulations! You can download all data in analysis_results.tar.gz file.**